# Intoduction
From the model comparison notebook, the results were:

- Ridge: RMSE = 0.1411 ± 0.0224
- Lasso: RMSE = 0.1371 ± 0.0271
- ElasticNet: RMSE = 0.1501 ± 0.0254
- rfr: RMSE = 0.1429 ± 0.0083


ElasticNet has the worst RMSE, despite having added complexity than Ridge. Lasso has the worst uncertainty, indicating high instability. These models underperform at baseline levels, so they will not be involved in this notebook


In [20]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

In [21]:
import numpy as np
import pandas as pd
import joblib

from src.data import load_train_data
from src.features import add_engineered_features

data = load_train_data()
X = add_engineered_features(data)
y = np.log1p(data['SalePrice'])

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

nominal_cols = ['MSZoning','MSSubClass', 'Street',  'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType','Foundation', 'Heating','CentralAir', 'Electrical','Functional',
        'GarageFinish','PavedDrive',
       'SaleType', 'SaleCondition'
]

ordinal_cols = ['OverallQual','OverallCond','ExterQual', 'ExterCond',
                'HeatingQC','KitchenQual'

]
num_structured_missing = ['GarageYrBlt']
num_unstruc_missing = ['LotFrontage']

rest_num_cols = [
  col for col in X.columns
  if col not in nominal_cols + ordinal_cols + ["SalePrice"] + num_structured_missing + num_unstruc_missing
]

# pipelines

# for numerical attributes with structured missingness, value 0 will be imputed 
num_struc_missing_pipeline = Pipeline([
  ('imputer',
   SimpleImputer(strategy='constant',
   fill_value = 0)),
  ('scaler',StandardScaler())
])

# for numerical attributes with true missingness, and the rest of the num cols, the median value will be imputed 
rest_num = Pipeline([
  ('imputer',
   SimpleImputer(strategy='median')),
   ('scaler',StandardScaler())
])


# ordinal pipelines improved
num_ord_cols = ['OverallQual','OverallCond']
cat_ord_cols = ['ExterQual', 'ExterCond','HeatingQC','KitchenQual']
num_ordinal_pipeline = Pipeline([
  ('imputer', SimpleImputer(strategy='most_frequent')),
  ('scaler',StandardScaler())
])

qual_scale = ['None', 'Po','Fa','TA','Gd','Ex']
cat_ordinal_pipeline = Pipeline([
  ('imputer', SimpleImputer(strategy='most_frequent')),
  ('encoder', OrdinalEncoder(
    categories=[qual_scale] * len(cat_ord_cols),
    handle_unknown= 'use_encoded_value',
    unknown_value=-1
  ))
])

# preprocessor
preprocessor = ColumnTransformer([
  ('num_struc',num_struc_missing_pipeline,
    num_structured_missing),
  ('rest_num', rest_num, 
   rest_num_cols),
  ('nominal', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_cols),
  ('num_ordinal', num_ordinal_pipeline, num_ord_cols),
  ('cat_ordinal', cat_ordinal_pipeline, cat_ord_cols),
])

At later error analysis stage, an error raised up becuase of the previous manual encoding, here I have improve the code such that it is part of the pipeline

# GridSearch on Ridge

In [22]:
from sklearn.model_selection import GridSearchCV

pipeline_ridge = Pipeline([
  ('preprocessor', preprocessor),
  ('model',Ridge())
])

param_grid_ridge = {
  'model__alpha' : [0.01, 0.1, 1.0, 10.0, 100.0] #in magnitudes
}

grid_ridge = GridSearchCV(
  estimator=pipeline_ridge,
  param_grid=param_grid_ridge,

  scoring="neg_root_mean_squared_error",
  cv=5,
  n_jobs=-1
)

grid_ridge.fit(X,y)

print("Best alpha:", grid_ridge.best_params_)
print("Best RMSE:",-grid_ridge.best_score_)

Best alpha: {'model__alpha': 10.0}
Best RMSE: 0.13657573554873276


In [23]:
# extract the best model
best_ridge = grid_ridge.best_estimator_

# Random Forest

In [24]:
from sklearn.ensemble import RandomForestRegressor

pipeline_rfr = Pipeline([
  ('preprocessor', preprocessor),
  ('model',RandomForestRegressor(random_state=42))
])

param_grid_rfr = {
  'model__n_estimators': [200,500],
  'model__max_depth': [None, 10,20],
  'model__min_samples_leaf': [1,5,10]
}


grid_rfr = GridSearchCV(
  estimator=pipeline_rfr,
  param_grid = param_grid_rfr,
  scoring= "neg_root_mean_squared_error",
  cv = 5,
  n_jobs= -1
)

grid_rfr.fit(X,y)

print("Best alpha:", grid_rfr.best_params_)
print("Best RMSE:",-grid_rfr.best_score_)

Best alpha: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 500}
Best RMSE: 0.1424116719610545


In the previous notebook I did see potential in non-linear models, and I only used one non-linear model. Although random forest underperformed in this case, it might be due to the lack of boosting. 

I will try another more expressive non-linear model - LightGBM, known for strong performance on structured data to ensure fairness

I will use RandomizedSearchCV instead of GridSearchCv, due to larger hyperparameter space, and for higher computational efficiency 

In [25]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV

import numpy as np 

pipeline_lgb = Pipeline([
  ('preprocessor', preprocessor),
  ('model',LGBMRegressor(random_state=42))
])

param_dist = {
  'model__n_estimators': [100, 300, 500],
  'model__learning_rate': [0.01, 0.05, 0.1],
  'model__num_leaves': [31, 63, 127],
  'model__max_depth': [-1,10,20],
  'model__min_child_samples':[5,10,20] 
}

lgb_search = RandomizedSearchCV(
  estimator = pipeline_lgb,
  param_distributions=param_dist, 
  n_iter=30,

scoring = 'neg_root_mean_squared_error',
cv=5,
n_jobs= -1,
random_state=42,
verbose=1
)

lgb_search.fit(X,y)

print('Best LGB params:', lgb_search.best_params_)
print('Best LGB CV RMSE:', -lgb_search.best_score_)


Fitting 5 folds for each of 30 candidates, totalling 150 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002510 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2927
[LightGBM] [Info] Number of data points in the train set: 1460, number of used features: 199
[LightGBM] [Info] Start training from score 12.024057
Best LGB params: {'model__num_leaves': 31, 'model__n_estimators': 500, 'model__min_child_samples': 5, 'model__max_depth': -1, 'model__learning_rate': 0.01}
Best LGB CV RMSE: 0.13096131792884752


# Results

In [26]:
def extract_cv_stats(search_object, name):
  mean_rmse = -search_object.best_score_
  std_rmse = search_object.cv_results_['std_test_score'][search_object.best_index_]
  return {'model':name, 'Mean_RMSE':mean_rmse, 'std_RMSE': std_rmse}

results = pd.DataFrame([
  extract_cv_stats(grid_ridge,"Ridge"),
  extract_cv_stats(grid_rfr,"RandomForest"),
  extract_cv_stats(lgb_search,"LightGBM"),
])

print(results)

          model  Mean_RMSE  std_RMSE
0         Ridge   0.136576  0.023301
1  RandomForest   0.142412  0.008456
2      LightGBM   0.130961  0.010407


# Save final models

In [27]:
MODELS_DIR = ROOT / "models"

joblib.dump(grid_ridge.best_estimator_, MODELS_DIR / "ridge_final.pkl")
joblib.dump(grid_rfr.best_estimator_, MODELS_DIR / "rfr_final.pkl")
joblib.dump(lgb_search.best_estimator_, MODELS_DIR / "lgbm_final.pkl")

print('models saved to models/')

models saved to models/
